In [22]:
from ast import *
from utils import *

In [23]:
Binding = tuple[Name, expr]
Temporaries = list[Binding]

In [24]:

def rco_exp(e, need_atomic) -> tuple[e,Temporaries]:
        # YOUR CODE HERE
        match e:
            case Constant(value):
                return (Constant(value), [])
            case Name(id):
                  return (Name(id), [])
            case UnaryOp(USub(),v):
                  new_v , bindings = rco_exp(v,True)
                  new_e = UnaryOp(USub(),new_v)
                  if need_atomic:
                        temp_name = generate_name("temp")
                        return (Name(temp_name), bindings + [(temp_name,new_e)])
                  else:
                        return (new_e, bindings)
            case BinOp(left, Add(),right):
                    new_l , b1 = rco_exp(left,True)
                    new_r, b2 = rco_exp(right,True)
                    new_e = BinOp(new_l, Add(),new_r)
                    if need_atomic:
                          temp_name = generate_name("temp")
                          return (Name(temp_name), b1 + b2 + [(temp_name,new_e)])
                    else:
                          return (new_e, b1 + b2)

            case BinOp(left, Sub(), right):
                    new_l, b1 = rco_exp(left,True)
                    new_r, b2 = rco_exp(right,True)
                    new_e = BinOp(new_l, Sub(), new_r)
                    if need_atomic:
                          temp_name = generate_name("temp")
                          return (Name(temp_name), b1 + b2 + [(temp_name,new_e)])
                    else:
                          return (new_e , b1 + b2)

            case Call(Name('input_int'),[]):
                    new_e = Call(Name('input_int'),[])
                    if need_atomic:
                          temp_name = generate_name("temp")
                          return (Name(temp_name), [(temp_name,new_e)])
                    else:
                          return (new_e, [])
                    
              
                
       
            
        pass 

In [25]:
def rco_stmt(s: stmt) -> List[stmt]:
        # YOUR CODE HERE
        match s:

            case Assign([Name(var)],v):
                value, bindings = rco_exp(v,False)
                stmts = [Assign([Name(t)],e) for (t,e) in bindings]
                return stmts + [Assign([Name(var)], value)]
             
            case Expr(Call(Name('print'),[arg])):
                value, bindings = rco_exp(arg,True)
                stmts = [Assign([Name(t)],e) for (t,e) in bindings]
                return stmts + [Expr(Call(Name('print'),[value]))]
             
            case Expr(exp):
                value, bindings = rco_exp(exp,False)
                stmts = [Assign([Name(t)],e) for (t,e) in bindings]
                return stmts + [Expr(value)]
       
          

        pass        

In [26]:
def remove_complex_operands(p: Module) -> Module:
        # YOUR CODE HERE
        match p:
            case Module(body):
                new_body = []
                for stmt in body:
                     new_body.extend(rco_stmt(stmt))
                return Module(new_body)
        pass         

In [27]:


if __name__ == "__main__":
    import textwrap
    code = textwrap.dedent("""
    x = 5 + (-4-3)
    y = x + 10
    print(y)""")
    parsed_code = parse(code)
    transformed_code = remove_complex_operands(parsed_code)
    print(transformed_code)
    

    temp.2 = -(4)    temp.3 = (temp.2 - 3)    x = (5 + temp.3)    y = (x + 10)    print(y)
